# CREMA-D -> spectrogrammes RGB + CSV de metadonnees

Telecharge CREMA-D via kagglehub (deja au format AudioWAV/, pas de souci git-lfs),
convertit chaque wav en segments de 3s en mel-spectrogramme RGB, et ajoute ses
metadonnees (age/genre via VideoDemographics.csv, confiance via l'accord des auditeurs
audio de tabulatedVotes.csv, si ces fichiers sont presents dans le mirroir Kaggle) a la
suite dans le meme `data/labels.csv` et `data/spectrograms/labels.csv` que les autres bases.

Prerequis : un compte Kaggle avec un token API configure (`~/.kaggle/kaggle.json` ou
variables d'environnement KAGGLE_USERNAME/KAGGLE_KEY), sinon `kagglehub` demande une
authentification interactive au premier appel.

In [ ]:
%pip install -q librosa matplotlib numpy pandas soundfile pillow nbstripout kagglehub

import sys
!{sys.executable} -m nbstripout --install --attributes .gitattributes

In [ ]:
import io, zipfile, urllib.request
from pathlib import Path

import kagglehub
import librosa
import numpy as np
import pandas as pd
import soundfile as sf
from matplotlib import cm
from PIL import Image

META_URL = "https://github.com/CheyneyComputerScience/CREMA-D/archive/refs/heads/master.zip"
DATA_DIR = Path("data")
META_DIR = DATA_DIR / "CREMA-D-master"
OUT = Path("data/spectrograms")
LABELS_CSV = Path("data/labels.csv")

DATASET_NAME = "CREMA-D"

# colonnes communes a toutes les bases (union des taxonomies)
EMOTIONS_ALL = [
    "anger", "boredom", "calm", "disgust", "fear",
    "happiness", "neutral", "sadness", "surprise",
]
CREMAD_EMOTIONS = {"anger", "disgust", "fear", "happiness", "neutral", "sadness"}

# 3eme partie du nom de fichier (emotion)
EMOTION_CODES = {
    "ANG": "anger", "DIS": "disgust", "FEA": "fear",
    "HAP": "happiness", "NEU": "neutral", "SAD": "sadness",
}

In [ ]:
# 1. audio (kagglehub, deja au format wav) + metadonnees (zip github, VideoDemographics/tabulatedVotes)
RAW = Path(kagglehub.dataset_download("ejlok1/cremad"))
print("RAW =", RAW)

audio_dirs = list(RAW.rglob("AudioWAV"))
audio_dir = audio_dirs[0] if audio_dirs else RAW
wavs = sorted(audio_dir.glob("*.wav"))
print(len(wavs), "fichiers wav")

if not META_DIR.exists():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print("downloading metadata...")
    data = urllib.request.urlopen(META_URL).read()
    zipfile.ZipFile(io.BytesIO(data)).extractall(DATA_DIR)

# debug : liste les CSV/txt trouves dans le zip de metadonnees
print("contenu de META_DIR (csv/txt) :")
for p in sorted(META_DIR.rglob("*")):
    if p.is_file() and p.suffix.lower() in {".csv", ".txt"}:
        print(" ", p.relative_to(META_DIR))

In [ ]:
# 2. metadonnees (nom de fichier + demographie + accord des auditeurs) -> 1 CSV (accumule sur plusieurs bases)
# demographie et votes : optionnels, pas garantis presents dans le mirroir Kaggle
demo = None
demo_files = list(META_DIR.rglob("VideoDemographics.csv"))
print("VideoDemographics.csv trouve :", demo_files)
if demo_files:
    demo = pd.read_csv(demo_files[0])
    demo["ActorID"] = demo["ActorID"].astype(int)
    demo["Sex"] = demo["Sex"].str.lower()
    print("demo : colonnes", list(demo.columns), "-", len(demo), "lignes")

votes = None
votes_files = list(META_DIR.rglob("tabulatedVotes.csv"))
print("tabulatedVotes.csv trouve :", votes_files)
if votes_files:
    votes = pd.read_csv(votes_files[0])
    votes = votes[votes["Unnamed: 0"].astype(str).str.startswith("1")]  # prefixe 1 = votes audio seul (voice)
    votes = votes.copy()
    votes["file"] = votes["fileName"] + ".wav"
    print("votes (voice) : colonnes", list(votes.columns), "-", len(votes), "lignes")

rows = []
for wav in wavs:
    actor, sentence, emo_code, level = wav.stem.split("_")
    rows.append({
        "file": wav.name,
        "subdir": str(wav.relative_to(RAW).parent),
        "speaker": int(actor),
        "language": "eng",
        "transcription": sentence,
        "emotion": EMOTION_CODES[emo_code],
        "duration": round(sf.info(wav).duration, 3),
    })
df = pd.DataFrame(rows)

if demo is not None:
    df = df.merge(demo[["ActorID", "Age", "Sex"]], left_on="speaker", right_on="ActorID", how="left")
    df = df.drop(columns=["ActorID"]).rename(columns={"Age": "age", "Sex": "gender"})
    print("age/gender manquants apres jointure :", df["age"].isna().sum(), "/", len(df))
else:
    df["age"] = None
    df["gender"] = None

if votes is not None:
    df = df.merge(votes[["file", "agreement"]], on="file", how="left")
    df = df.rename(columns={"agreement": "emotion.confidence"})
    print("emotion.confidence manquants apres jointure :", df["emotion.confidence"].isna().sum(), "/", len(df))
else:
    df["emotion.confidence"] = None
df["emotion.naturalness"] = None

df.insert(0, "dataset", DATASET_NAME)
df.insert(0, "root", str(RAW))

if LABELS_CSV.exists():
    existing = pd.read_csv(LABELS_CSV, na_values=["None"])
    dup = existing.merge(df[["root", "file"]], on=["root", "file"], how="inner")
    if len(dup):
        raise ValueError(f"{len(dup)} fichiers de {DATASET_NAME} deja presents dans {LABELS_CSV}")
    same = existing[existing["dataset"] == DATASET_NAME]
    prior_speaker = dict(zip(same["initial_speaker"], same["speaker"]))
    prior_transcript = dict(zip(same["initial_transcript"], same["transcript"]))
    next_speaker = int(existing["speaker"].max()) + 1
    next_transcript = int(existing["transcript"].max()) + 1
else:
    existing = None
    prior_speaker = {}
    prior_transcript = {}
    next_speaker = 1
    next_transcript = 1

df["initial_speaker"] = df["speaker"]
speaker_map = {}
for code in sorted(df["speaker"].unique()):
    if code in prior_speaker:
        speaker_map[code] = prior_speaker[code]
    else:
        speaker_map[code] = next_speaker
        next_speaker += 1
df["speaker"] = df["speaker"].map(speaker_map)

df["initial_transcript"] = df["transcription"]
transcript_map = {}
for code in sorted(df["transcription"].unique()):
    if code in prior_transcript:
        transcript_map[code] = prior_transcript[code]
    else:
        transcript_map[code] = next_transcript
        next_transcript += 1
df["transcript"] = df["transcription"].map(transcript_map)
df = df.drop(columns=["transcription"])

for emo in EMOTIONS_ALL:
    df[emo] = (df["emotion"] == emo).astype(int) if emo in CREMAD_EMOTIONS else None
df = df.drop(columns=["emotion"])

cols = ["root", "dataset", "subdir", "file", "duration", "speaker", "initial_speaker",
        "age", "gender", "language", "emotion.confidence", "emotion.naturalness",
        "transcript", "initial_transcript"] + EMOTIONS_ALL
df = df[cols]

if existing is not None:
    df = pd.concat([existing, df], ignore_index=True)

df.to_csv(LABELS_CSV, index=False, na_rep="None")
print(len(df), "lignes dans", LABELS_CSV)

In [ ]:
# 3. wav -> segments de 3s -> mel-spectrogramme -> image RGB (accumule sur plusieurs bases)
SR = 16000
DURATION = 3.0
MIN_SEG = 1.0
N_FFT = int(0.025 * SR)
HOP = int(0.010 * SR)
N_MELS = 64
N_FRAMES = int(DURATION * SR / HOP)
CHUNK = int(DURATION * SR)
MIN_SAMPLES = int(MIN_SEG * SR)

SPEC_CSV = OUT / "labels.csv"
label_by_file = dict(zip(df["file"], df[EMOTIONS_ALL].fillna(0).idxmax(axis=1)))
meta = df.set_index("file").to_dict(orient="index")

spec_existing = pd.read_csv(SPEC_CSV, na_values=["None"]) if SPEC_CSV.exists() else None
done = set(zip(spec_existing["root"], spec_existing["base_file"])) if spec_existing is not None else set()

def mel_to_rgb(y):
    y = librosa.util.fix_length(y, size=CHUNK)
    mel = librosa.feature.melspectrogram(
        y=y, sr=SR, n_fft=N_FFT, hop_length=HOP, win_length=N_FFT, n_mels=N_MELS
    )
    db = librosa.power_to_db(mel)[:, :N_FRAMES]
    z = (db - db.mean()) / (db.std() + 1e-9)
    norm = np.clip(z, -3, 3) / 6 + 0.5
    rgb = (cm.viridis(norm[::-1])[..., :3] * 255).astype(np.uint8)
    return Image.fromarray(rgb)

rows = []
for wav in wavs:
    if (str(RAW), wav.name) in done:
        continue
    y, _ = librosa.load(wav, sr=SR, mono=True)
    label = label_by_file[wav.name]

    for n, start in enumerate(range(0, len(y), CHUNK)):
        seg = y[start:start + CHUNK]
        if len(seg) < MIN_SAMPLES:
            continue
        name = f"{wav.stem}_{n}"
        dest = OUT / label
        dest.mkdir(parents=True, exist_ok=True)
        mel_to_rgb(seg).save(dest / f"{name}.png")
        rows.append({
            "spectrogram": f"{label}/{name}.png",
            "base_file": wav.name,
            "chunk": n,
            **meta[wav.name],
            "duration": round(sf.info(wav).duration, 3),
        })

new_spec = pd.DataFrame(rows)
spec_df = pd.concat([spec_existing, new_spec], ignore_index=True) if spec_existing is not None else new_spec
spec_df.to_csv(SPEC_CSV, index=False, na_rep="None")
print(len(rows), "nouveaux spectrogrammes,", len(spec_df), "total dans", SPEC_CSV)

In [ ]:
# 4. verification rapide
for d in sorted(OUT.iterdir()):
    if d.is_dir():
        print(d.name, len(list(d.glob("*.png"))))

display(Image.open(next(OUT.rglob("*.png"))))

import csv as _csv
with LABELS_CSV.open(encoding="utf-8") as f:
    for i, row in zip(range(5), _csv.reader(f)):
        print(row)